# Phonetic-Convergence Proj

이 프로젝트는 음성수렴 현상을 관찰하기 위한 실험입니다.
분석에 관한 코드 및 실행 결과를 정리한 notebook입니다.

## Module import

In [1]:
import parselmouth
from parselmouth import praat
from parselmouth.praat import call
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
#import librosa
#import librosa.display
import sounddevice as sd
from datetime import datetime
from tqdm import tqdm
import seaborn as sns
import matplotlib.patches as patches
from pathlib import Path
from pprint import pprint

In [2]:
# User define module
from enhanced_formant_analysis import EnhancedFormantAnalyzer
from textgrid_io import TextGridReader
from word_tier_generator import WordTierGenerator

## Hyperparameter

In [3]:
F1_MIN, F1_MAX = 200, 1200    # F1 범위 (Hz)
F2_MIN, F2_MAX = 500, 3000   # F2 범위 (Hz)

In [4]:
# 소스 코드 기준 상대 경로를 이용하여 proj_dir 위치 설정.
proj_dir = os.path.join(Path.home(), 'Documents', 'WorkSpace', 'phoneticConvergence')
exp_data_dir = os.path.join(proj_dir, 'data', 'experiment_data')
print(f"proj_dir: {proj_dir}")
print(f"exp_data_dir: {exp_data_dir}")

proj_dir: /Users/bagjuhyeon/Documents/WorkSpace/phoneticConvergence
exp_data_dir: /Users/bagjuhyeon/Documents/WorkSpace/phoneticConvergence/data/experiment_data


In [5]:
# wav_dir, annotation_dir 위치 지정.
model_wav_dir = os.path.join(exp_data_dir, 'model_talker', 'KFA_audio')
model_annotation_dir = os.path.join(exp_data_dir, 'model_talker', 'KFA_annotation_edited')

# 파일 리스트 가져오기. (확장자 고려 필수!)
model_audio_list = [file for file in os.listdir(model_wav_dir) if file.endswith(".wav")]
model_annotation_list = [file for file in os.listdir(model_annotation_dir) if file.endswith(".TextGrid")]
model_audio_list.sort()
model_annotation_list.sort()

# 리스트 확인
print(f"model_audio: {len(model_audio_list)}개")
print(f"model_annotation: {len(model_annotation_list)}개")
pprint(f"model_audio_list: {model_audio_list}")
pprint(f"model_annotation_list: {model_annotation_list}")

model_audio: 50개
model_annotation: 50개
("model_audio_list: ['난색.wav', '난파.wav', '내빈.wav', '내사.wav', "
 "'녹각.wav', '논개.wav', '누룩.wav', '누진.wav', '닌자.wav', "
 "'닝닝.wav', '말미.wav', '망루.wav', '매복.wav', '맹호.wav', "
 "'몰딩.wav', '몽매.wav', '무골.wav', '무운.wav', '미비.wav', "
 "'밀사.wav', '박애.wav', '발탁.wav', '백태.wav', '뱃심.wav', "
 "'복시.wav', '봉독.wav', '분개.wav', '불모.wav', '비재.wav', "
 "'빈축.wav', '상기.wav', '상록.wav', '새참.wav', '샐쭉.wav', "
 "'소조.wav', '송부.wav', '수하.wav', '술책.wav', '시문.wav', "
 "'실족.wav', '파군.wav', '판본.wav', '패물.wav', '팽배.wav', "
 "'포문.wav', '폭압.wav', '품띠.wav', '풍산.wav', '피폭.wav', "
 "'필생.wav']")
("model_annotation_list: ['난색.TextGrid', '난파.TextGrid', "
 "'내빈.TextGrid', '내사.TextGrid', '녹각.TextGrid', '논개.TextGrid', "
 "'누룩.TextGrid', '누진.TextGrid', '닌자.TextGrid', '닝닝.TextGrid', "
 "'말미.TextGrid',

In [6]:
# wav_dir, annotation_dir 위치 지정.
participants_dir = os.path.join(exp_data_dir,'results-KFA-reviewed')
participant_list = [file for file in os.listdir(participants_dir) if file.startswith("participant_LY")]
participant_list.sort()

# 리스트 확인
print(f"participant_list: {len(participant_list)}개")
pprint(f"participant_list: {participant_list}")

participant_list: 48개
("participant_list: ['participant_LY002', 'participant_LY004', "
 "'participant_LY005', 'participant_LY006', 'participant_LY007', "
 "'participant_LY008', 'participant_LY009', 'participant_LY010', "
 "'participant_LY011', 'participant_LY013', 'participant_LY015', "
 "'participant_LY017', 'participant_LY019', 'participant_LY020', "
 "'participant_LY021', 'participant_LY022', 'participant_LY023', "
 "'participant_LY024', 'participant_LY025', 'participant_LY026', "
 "'participant_LY027', 'participant_LY028', 'participant_LY029', "
 "'participant_LY030', 'participant_LY031', 'participant_LY032', "
 "'participant_LY033', 'participant_LY034', 'participant_LY035', "
 "'participant_LY036', 'participant_LY037', 'participant_LY038', "
 "'participant_LY039', 'participant_LY040', 'participant_LY041', "
 "'participant_LY042', 'participant_LY044', 'participant_LY045', "
 "'participant_LY046', 'participant_LY047', 'participant_LY048', "
 "'participant_LY050', 'participant_LY051'

## File Validity Check

### File Naming Check

model speaker의 음성 파일 리스트 naming check.

In [7]:
# any를 사용해서 wav 파일들 중 TextGrid랑 pair가 맞지 않는 경우가 있는지 check.
any([True if wav_file_name.replace('.wav', '.TextGrid') not in model_annotation_list else False for wav_file_name in model_audio_list])

False

participant 폴더에서 각 파일들의 naming check

In [8]:
# 모든 참가자 파일에 wav, annotation 파일이 맞게 존재하는지 확인.
for participant_id in tqdm(participant_list, desc="Checking validity of participant files"):
    participant_dir = os.path.join(participants_dir, participant_id)
    
    participant_wav_list = [file for file in os.listdir(participant_dir) if file.endswith(".wav")]
    participant_annotation_list = [file for file in os.listdir(participant_dir) if file.endswith(".TextGrid")]
    
    participant_wav_list.sort()
    participant_annotation_list.sort()

    # 각 참가자 당 wav 4개, textgrid 4개가 존재해야 하고, 이름이 동일해야 함.
    if len(participant_wav_list) != 4:
        raise FileNotFoundError(f"Not enough wav files in participant_dir: {participant_dir}")
    if len(participant_annotation_list) != 4:
        raise FileNotFoundError(f"Not enough textgrid files in participant_dir: {participant_dir}")
    
    # 각 참가자 당 wav 4개, textgrid 4개가 존재해야 하고, 이름이 동일해야 함.
    if any([True if wav_file_name.replace('.wav', '.TextGrid') not in participant_annotation_list else False for wav_file_name in participant_wav_list]):
        raise FileNotFoundError(f"wav and textgrid files are not paired in participant_dir: {participant_dir}.")

print("All files are paired correctly.")


Checking validity of participant files: 100%|██████████| 48/48 [00:00<00:00, 17945.15it/s]

All files are paired correctly.


### Boundary Fixation

In [9]:
# 모든 참가자 textgrid 파일에 boundary fixation.
for participant_id in tqdm(participant_list, desc="Checking validity of participant files"):
    participant_dir = os.path.join(participants_dir, participant_id)
    
    participant_annotation_list = [file for file in os.listdir(participant_dir) if file.endswith(".TextGrid")]
    participant_annotation_list.sort()

    for annotation_file in participant_annotation_list:
        annotation_path = os.path.join(participant_dir, annotation_file)
        # boundary fix
        tg = TextGridReader(annotation_path, fix_boundary_integrity=False, verbose=False)
        tg.boundary_fix(backup=True, inplace=True)

        # redefine Word tier 
        generator = WordTierGenerator(annotation_path, verbose=False)
        generator.add_word_tier_to_textgrid(inplace=True)

    
print("All boundaries of TextGrid files are fixed.")

Checking validity of participant files: 100%|██████████| 48/48 [00:01<00:00, 32.79it/s]

All boundaries of TextGrid files are fixed.


### Word, Phone tier numbering check

textgrid 파일 내의 word, phone tier랑 매칭. phone을 모아서 단어 만들면 그 단어가 word tier에 제대로 존재하는지 check.

뱃심 단어가 문제가 있어 이를 수정.
기존에는 phone tier에 "baedsim"이지만, word_tier에는 "baessim"으로 되어 있어. 모두 "baedsim"으로 통일.

In [ ]:
for model_annotation_file in tqdm(model_annotation_list, desc="Checking validity of model speaker files"):
    model_annotation_path = os.path.join(model_annotation_dir, model_annotation_file)
    tg = TextGridReader(model_annotation_path)

    # tier가 2개 인지 check
    if 'phone' not in tg.get_tier_names() or 'word' not in tg.get_tier_names():
        raise TypeError(f"phone or word tier is not found in the TextGrid file.{model_annotation_file}")
    
    # phone tier에서 phone fragment를 추출해서 붙이고, word tier랑 비교.
    phone_fragment = ""
    for interval in tg.get_intervals_by_tier('phone'):
        if interval.text != 'sp':
            phone_fragment += interval.text
    
    word_fragment = ""
    for word_interval in tg.get_intervals_by_tier('word'):
        if word_interval.text != 'sp':
            word_fragment += word_interval.text
    
    if phone_fragment.lower() != word_fragment.lower():
        raise ValueError(f"phones and word are not matched in the TextGrid file.{model_annotation_file}")

print("The word and phone tiers in all model speaker files are matched correctly.")


Checking validity of model speaker files: 100%|██████████| 50/50 [00:00<00:00, 8619.26it/s]

The word and phone tiers in all model speaker files are matched correctly.


참가자 파일도 마찬가지로 check
하다가보면, phone_tier 같은데에서 phone 끝에 spacebar 들어간 경우가 종종 있으니 확인 요망.

In [15]:
from collections import deque

In [19]:
# 모든 참가자의 textgrid에서 phone, word tier에서 phone 조각들이 제대로 되어 있나 check.
for participant_id in tqdm(participant_list, desc="Checking validity of participant files"):
    participant_dir = os.path.join(participants_dir, participant_id)
    
    participant_annotation_list = [file for file in os.listdir(participant_dir) if file.endswith(".TextGrid")]
    participant_annotation_list.sort()

    for participant_annotation_file in participant_annotation_list:
        participant_annotation_path = os.path.join(participant_dir, participant_annotation_file)
        tg = TextGridReader(participant_annotation_path)

        if 'phone' not in tg.get_tier_names() or 'word' not in tg.get_tier_names():
            raise TypeError(f"phone or word tier is not found in the TextGrid file.{participant_annotation_file}")

        phone_fragment_Q = deque()
        phone_fragment = ""
        for interval in tg.get_intervals_by_tier('phone'):
            if interval.text != 'sp':
                phone_fragment += interval.text
            elif interval.text == 'sp' and phone_fragment != "":
                phone_fragment_Q.append(phone_fragment)
                phone_fragment_Q.append("sp")
                phone_fragment = ""
            else:
                phone_fragment_Q.append("sp")

        for word_interval in tg.get_intervals_by_tier('word'):
            
            if len(phone_fragment_Q) > 0:
                Q_val = phone_fragment_Q.popleft()
            else:
                print(f"phones and word are not matched in the TextGrid file.{participant_annotation_file}")
                print(f"word_interval.text: {word_interval.text}")
                print(f"word_interval: {word_interval.xmin} ~ {word_interval.xmax}")
            
            if word_interval.text == Q_val:
                pass
            else:
                print(f"phones and word are not matched in the TextGrid file.{participant_annotation_file}")
                print(f"Q_val: {Q_val}")
                print(f"word_interval.text: {word_interval.text}")
                print(f"word_interval: {word_interval.xmin} ~ {word_interval.xmax}")
                raise ValueError(f"phones and word are not matched in the TextGrid file.{participant_annotation_file}")
    
print("All files are paired correctly.")


Checking validity of participant files:   8%|▊         | 4/48 [00:00<00:00, 83.64it/s]

phones and word are not matched in the TextGrid file.7_stage4_20250403_1946.TextGrid
word_interval.text: saecam
word_interval: 242.8525 ~ 243.3725
phones and word are not matched in the TextGrid file.7_stage4_20250403_1946.TextGrid
Q_val: sp
word_interval.text: saecam
word_interval: 242.8525 ~ 243.3725


ValueError: phones and word are not matched in the TextGrid file.7_stage4_20250403_1946.TextGrid

In [ ]:
from collections import deque

In [ ]:
# 모든 참가자의 textgrid에서 phone, word tier에서 phone 조각들이 제대로 되어 있나 check.
for participant_id in tqdm(participant_list, desc="Checking validity of participant files"):
    participant_dir = os.path.join(participants_dir, participant_id)
    
    participant_annotation_list = [file for file in os.listdir(participant_dir) if file.endswith(".TextGrid")]
    participant_annotation_list.sort()

    for participant_annotation_file in participant_annotation_list:
        participant_annotation_path = os.path.join(participant_dir, participant_annotation_file)
        tg = TextGridReader(participant_annotation_path)
        
        word_tier_Q = queue.Queue()

        for word_interval in tg.get_intervals_by_tier('word'):
            if word_interval.text != 'sp':
                word_tier_Q.put(word_interval.text)

        if 'stage2' in participant_annotation_file or 'stage6' in participant_annotation_file:
            correct_word_stimuli_num = 50
        elif 'stage4' in participant_annotation_file or 'stage5' in participant_annotation_file:
            correct_word_stimuli_num = 150
        else:
            raise ValueError(f"stage number is not valid. TextGrid file: {participant_annotation_file}")

        # 이 부분이 개수가 불일치 조건.
        if word_tier_Q.qsize() != correct_word_stimuli_num:
            print(f"missing word in the TextGrid file.{participant_annotation_file}")
            print(f"current word count / expected word stimuli number: {word_tier_Q.qsize()} / {correct_word_stimuli_num}")
            #raise ValueError(f"missing word in the TextGrid file.{participant_annotation_file}")

            word_tier_Q.


        else:
            pass
            #print(f"word tier is matched in the TextGrid file.{participant_annotation_file}")
    
#print("All files are paired correctly.")


Checking validity of participant files:  23%|██▎       | 11/48 [00:00<00:00, 101.92it/s]

missing word in the TextGrid file.02_stage4_20250404_1351.TextGrid
expected word stimuli number: 150
current word count: 151
missing word in the TextGrid file.6_stage5_20250404_1925.TextGrid
expected word stimuli number: 150
current word count: 151
missing word in the TextGrid file.7_stage5_20250403_1951.TextGrid
expected word stimuli number: 150
current word count: 149
missing word in the TextGrid file.LY15_stage5_20250410_2053.TextGrid
expected word stimuli number: 150
current word count: 151
missing word in the TextGrid file.17_stage2_20250404_1639.TextGrid
expected word stimuli number: 50
current word count: 51
missing word in the TextGrid file.25_stage5_20250407_1930.TextGrid
expected word stimuli number: 150
current word count: 142
missing word in the TextGrid file.25_stage6_20250407_1938.TextGrid
expected word stimuli number: 50
current word count: 51
missing word in the TextGrid file.LY26_stage5_20250409_1648.TextGrid
expected word stimuli number: 150
current word count: 149


Checking validity of participant files:  67%|██████▋   | 32/48 [00:00<00:00, 98.40it/s] 

missing word in the TextGrid file.28_stage2_20250408_0942.TextGrid
expected word stimuli number: 50
current word count: 49
missing word in the TextGrid file.LY029_stage6_20250410_1643.TextGrid
expected word stimuli number: 50
current word count: 49
missing word in the TextGrid file.42_stage5_20250430_1522.TextGrid
expected word stimuli number: 150
current word count: 149


Checking validity of participant files: 100%|██████████| 48/48 [00:00<00:00, 99.88it/s]

missing word in the TextGrid file.55_stage5_20250605_1924.TextGrid
expected word stimuli number: 150
current word count: 149


## Formant Analysis